In [ ]:
from db.conf import create_db_engine, get_async_session
from dotenv import find_dotenv, load_dotenv

from db.models import AnnotationKind

load_dotenv(find_dotenv())
engine = create_db_engine()
db = get_async_session(engine)

In [ ]:
from datetime import datetime, timedelta
from uuid import UUID

from db.repositories.helpers import full_video_data
from db.repositories.videos import VideoRepository

async with db() as session:
  video_repo = VideoRepository(session)
  videos = await video_repo.get_video_by_topic(
      UUID("1f8c458c-acca-11f0-a342-33c9fb0ca500"),
      load_annotations=True,
      annotations_to_load=[AnnotationKind.label, AnnotationKind.synopsis, AnnotationKind.action, AnnotationKind.transcription],
      load_meta=True,
      max_videos=20
  )

  video_data = [full_video_data(v) for v in videos]

In [ ]:
from core.agents.common import TemplateManager, gemini_2_5_flash_lite, gemini_2_5_flash, gpt_5_nano, medium_effort_gpt_5
from core.agents.challenge import ChallengeGenAgent, ChallengeGenAgentRun


agent = ChallengeGenAgent(gpt_5_nano(), medium_effort_gpt_5(), TemplateManager())

In [ ]:
run_input = ChallengeGenAgentRun(topic="Fashion - Outfit Check", videos=video_data, languages=["en"])

result = await agent.run(run_input)
result